In [1]:
%pip install -U -q "torchao>=0.16.0" "peft>=0.10.0" "transformers>=4.40.0" datasets accelerate evaluate scikit-learn seaborn
%load_ext autoreload
%autoreload 2

# Now import your functions
from util import *
from util import save_custom_model
from transformers import DataCollatorWithPadding

# Install/Update specifically for 2026 SOTA requirements

# Sanity check: This should print "True" if step 1 was done correctly
import torch
print(f"GPU Active: {torch.cuda.is_available()}")

Note: you may need to restart the kernel to use updated packages.


/home/mhenheik/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU Active: True


In [2]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

file_path = './train_lang.csv'
dataset = load_dataset('csv', data_files=file_path, split='train')
 
#target_langs = {'eng_Latn'}

#dataset = dataset.filter(
#    lambda x: x['lang'] in target_langs
#)

#dataset = dataset.map(lambda x: {"label": (x["label"]) / 4})

# 3. Create the 90/10 split
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Filtered Dataset Size: {len(dataset['train']) + len(dataset['test'])}")

Filtered Dataset Size: 252000


In [3]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [4]:
from datasets import Value

def preprocess(examples):
    tokenized = tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)
    # Ensure they are floats here
    tokenized["label"] = [float(r) for r in examples["label"]]
    tokenized["lang"] = [0 if l == "eng_Latn" else 1 for l in examples["lang"]]
    return tokenized

# 1. Map the function
tokenized_ds = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)

# 2. CRITICAL: Explicitly cast the column to float32
# This prevents the library from converting them back to integers (Long)
tokenized_ds = tokenized_ds.cast_column("label", Value("float32"))
tokenized_ds = tokenized_ds.cast_column("lang", Value("int64"))

# 3. Double-check the format
tokenized_ds.set_format("torch")

In [5]:
import numpy as np
import evaluate
from sklearn.metrics import mean_absolute_error

metric = evaluate.load("mae")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.squeeze()
    
    # 1. Standardize the predictions to the 0-4 range
    # Ensure they don't go below 0 or above 4 before rounding
    preds_clipped = np.clip(predictions, 0, 4)
    
    # 2. Round to the nearest integer (this is what you submit)
    preds_rounded = np.rint(preds_clipped).astype(int)
    
    # 3. Calculate metrics
    # This MAE reflects your actual Kaggle performance
    mae_rounded = mean_absolute_error(labels, preds_rounded)
    
    # This MAE shows the underlying precision (useful for debugging)
    mae_raw = mean_absolute_error(labels, predictions)
    
    return {
        "mae": mae_rounded,
        "raw_mae": mae_raw
    }

In [11]:
from transformers import XLMRobertaModel, PreTrainedModel, XLMRobertaConfig, XLMRobertaPreTrainedModel
import torch.nn.functional as F

class XLMRobertaSentimentRegressor(nn.Module):
    def __init__(self, model_id, config, out_dim=1):
        super().__init__()
        self.config = config
        self.roberta = XLMRobertaModel.from_pretrained(model_id)
        hidden_size = config.hidden_size # 768 for base
        
        # A deeper MLP Head
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1)
        )
        
    def forward(self, input_ids, attention_mask, labels=None, lang=None, **kwargs):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use the [CLS] token representation (pooler_output or first token)
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        
        logits = self.regressor(pooled_output)
        
        # --- RANGE SQUASHING ---
        # If your sentiment is 0 to 1: use torch.sigmoid(logits)
        # If your sentiment is -1 to 1: use torch.tanh(logits)
        # If your sentiment is 1 to 5: 1 + 4 * torch.sigmoid(logits)
        prediction = 4 * torch.sigmoid(logits) 
        
        return {"logits": prediction}

class XLMRobertaCircular(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
        # Output 2 values: [x, y] representing (cos, sin)
        hidden_size = config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 2)
        )
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
            outputs = self.roberta(input_ids, attention_mask=attention_mask)
            sequence_output = outputs[0][:, 0, :]
            
            # 1. Generate 2D coordinates
            coords = self.regressor(sequence_output) 
            
            # Normalize for directional math
            pred_vecs = torch.nn.functional.normalize(coords, p=2, dim=1)
            
            # 2. Convert to 0-4 scalar for logits (Inference/Eval)
            theta = torch.atan2(pred_vecs[:, 1], pred_vecs[:, 0])
            theta_positive = (theta + 2 * torch.pi) % (2 * torch.pi)
            scalar_output = (theta_positive / (2 * torch.pi)) * 4.0 
                    
            loss = None
            if labels is not None:
                # --- Standard Circular Loss ---
                target_angles = (labels.float() / 4.0) * 2 * torch.pi
                target_vecs = torch.stack([torch.cos(target_angles), torch.sin(target_angles)], dim=1)
                base_loss = (1 - (pred_vecs * target_vecs).sum(dim=1)).mean()
    
                # --- The Saturated Barrier (Regularization) ---
                # x_coords is 1.0 at the absolute 0/4 junction
                x_coords = pred_vecs[:, 0]
                
                # delta=0.2 means the penalty starts roughly at sentiment 3.8/0.2
                delta = 0.2 
                # Threshold needs to be on the same device as pred_vecs
                threshold = torch.cos(torch.tensor(delta, device=x_coords.device))
                
                # We use tanh to 'cut the head off' the penalty.
                # Once it reaches a certain point, the penalty flattens,
                # allowing the model to 'pop' through to the other side.
                strength = 10.0 # Controls the slope of the hill
                diff = torch.clamp(x_coords - threshold, min=0)
                gap_penalty = torch.tanh(strength * diff).mean()
                
                # Final combined loss
                # Lambda=0.8 is quite stiff; monitor if your 0s and 4s disappear
                loss = base_loss + (0.8 * gap_penalty)
    
            return {"loss": loss, "logits": scalar_output} if loss is not None else {"logits": scalar_output}


class XLMRobertaMobius(XLMRobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = XLMRobertaModel(config)
 
        hidden_size = config.hidden_size
        self.intensity_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1)
        )

        self.sarcasm_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size // 4),
            nn.GELU(),
            nn.Linear(hidden_size // 4, 1)
        )
        self.sarcasm_head[-1].bias.data.fill_(-3.0)
        self.post_init()
        self.loss_fct = nn.HuberLoss(delta=0.75)
        #self.loss_fct = nn.MSELoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        sequence_output = outputs[0][:, 0, :]
        
        # 1. Branching Heads
        # Intensity (P): How 'strong' is the language? (0-4)
        # We use a scaled sigmoid to ensure it stays in the label range
        intensity = torch.sigmoid(self.intensity_head(sequence_output)) * 4.0
        
        # Incongruity (S): Is this sarcasm? (0-1)
        sarcasm_prob = torch.sigmoid(self.sarcasm_head(sequence_output))
        
        # 2. The Möbius Calculation
        # (1 - S) * P + S * (4 - P)
        final_sentiment = (1 - sarcasm_prob) * intensity + sarcasm_prob * (4.0 - intensity)
        
        loss = None
        if labels is not None:
            # 3. Huber Loss (Smooth L1)
            # 'delta' here is the Huber threshold, usually 1.0
            loss = self.loss_fct(final_sentiment.view(-1), labels.float().view(-1))
            
        return {"loss": loss, "logits": final_sentiment}
        
class PassthroughCollator:
    def __init__(self):
        self.printed = False

    def __call__(self, features):
        # 1. Print exactly what arrived on the first batch
        if not self.printed:
            print("\n" + "="*50)
            print("🚨 DEBUG: WHAT ACTUALLY REACHED THE COLLATOR? 🚨")
            print(f"Keys present:  {list(features[0].keys())}")
            if "lang" in features[0]:
                print(f"'lang' value:  {features[0]['lang']} (Type: {type(features[0]['lang'])})")
            else:
                print("'lang' value:  MISSING (Trainer stripped it before collator)")
            print("="*50 + "\n")
            self.printed = True

        # 2. Blindly convert whatever keys exist into tensors
        batch = {}
        for key in features[0].keys():
            batch[key] = torch.stack([f[key] for f in features])
            
        return batch
    
class HuberTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs): # Added **kwargs
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        loss_fct = nn.HuberLoss(delta=0.1)
        loss = loss_fct(logits.squeeze(), labels.squeeze())
        
        return (loss, outputs) if return_outputs else loss

class PCGradHuberTrainer(HuberTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        
        # 1. Pop the tensor (it is now a GPU tensor of 0s and 1s)
        langs = inputs.pop("lang", None)
        if langs is None:
            raise KeyError("Still missing. Ensure remove_unused_columns=False is in TrainingArguments.")
            
        # 2. Find row indices for each language using tensor boolean masking
        en_idx = (langs == 0).nonzero(as_tuple=True)[0]
        de_idx = (langs == 1).nonzero(as_tuple=True)[0]
        
        # 3. SAFETY VALVE
        if len(en_idx) == 0 or len(de_idx) == 0:
            loss = self.compute_loss(model, inputs)
            self.accelerator.backward(loss)
            return loss.detach()

        # 4. Slice the tensors for each language
        inputs_en = {k: v[en_idx] for k, v in inputs.items()}
        inputs_de = {k: v[de_idx] for k, v in inputs.items()}
        
        params = [p for p in model.parameters() if p.requires_grad]
        
        # 5. Backward Pass A (English)
        model.zero_grad()
        loss_en = self.compute_loss(model, inputs_en)
        self.accelerator.backward(loss_en)
        grads_en = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]
        
        # 6. Backward Pass B (German)
        model.zero_grad()
        loss_de = self.compute_loss(model, inputs_de)
        self.accelerator.backward(loss_de)
        grads_de = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]
        
        # 7. Symmetric PCGrad Vector Surgery
        model.zero_grad()
        with torch.no_grad():
            for p, g_en, g_de in zip(params, grads_en, grads_de):
                g_en_flat, g_de_flat = g_en.flatten(), g_de.flatten()
                
                dot = torch.dot(g_en_flat, g_de_flat)
                
                if dot < 0:
                    norm_en_sq = torch.dot(g_en_flat, g_en_flat) + 1e-8
                    norm_de_sq = torch.dot(g_de_flat, g_de_flat) + 1e-8
                    
                    g_en_proj = g_en - (dot / norm_de_sq) * g_de
                    g_de_proj = g_de - (dot / norm_en_sq) * g_en
                    
                    p.grad = g_en_proj + g_de_proj
                else:
                    p.grad = g_en + g_de
                    
        return (loss_en + loss_de).detach() / 2.0

class GradVacTrainer(HuberTrainer):
    def __init__(self, *args, beta=0.9, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.rho_12 = 0.0

    def training_step(self, model, inputs, num_items_in_batch=None, **kwargs):
        model.train()
        langs = inputs.pop("lang", None)
        
        en_idx = (langs == 0).nonzero(as_tuple=True)[0]
        de_idx = (langs == 1).nonzero(as_tuple=True)[0]
        
        # Base case
        if len(en_idx) == 0 or len(de_idx) == 0:
            outputs = model(**inputs)
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
            self.accelerator.backward(loss)
            return loss.detach()

        inputs_en = {k: v[en_idx] for k, v in inputs.items()}
        inputs_de = {k: v[de_idx] for k, v in inputs.items()}
        params = [p for p in model.parameters() if p.requires_grad]

        # 1. Get Gradients for Task A (EN)
        model.zero_grad()
        outputs_en = model(**inputs_en)
        loss_en = outputs_en["loss"] if isinstance(outputs_en, dict) else outputs_en.loss
        self.accelerator.backward(loss_en)
        grads_en = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]

        # 2. Get Gradients for Task B (DE)
        model.zero_grad()
        outputs_de = model(**inputs_de)
        loss_de = outputs_de["loss"] if isinstance(outputs_de, dict) else outputs_de.loss
        self.accelerator.backward(loss_de)
        grads_de = [p.grad.clone() if p.grad is not None else torch.zeros_like(p) for p in params]

        # 3. GradVac Surgery
        model.zero_grad()
        with torch.no_grad():
            g1_all = torch.cat([g.flatten() for g in grads_en])
            g2_all = torch.cat([g.flatten() for g in grads_de])
            
            norm1, norm2 = torch.norm(g1_all), torch.norm(g2_all)
            cos_theta = torch.dot(g1_all, g2_all) / (norm1 * norm2 + 1e-8)
            
            self.rho_12 = self.beta * self.rho_12 + (1 - self.beta) * cos_theta.item()
            
            if cos_theta < self.rho_12:
                sin_theta = torch.sqrt(1 - cos_theta**2 + 1e-8)
                target_sin = torch.sqrt(1 - torch.tensor(self.rho_12)**2 + 1e-8)
                
                phi = (norm1 * (self.rho_12 * sin_theta - cos_theta * target_sin)) / (norm2 * target_sin + 1e-8)
                
                for p, g_en, g_de in zip(params, grads_en, grads_de):
                    p.grad = (g_en + phi * g_de) + g_de
            else:
                for p, g_en, g_de in zip(params, grads_en, grads_de):
                    p.grad = g_en + g_de
                    
        return (loss_en + loss_de).detach() / 2.0

In [12]:
#model = AutoModelForSequenceClassification.from_pretrained(
#        model_id, 
#        num_labels=1, 
#        problem_type="regression"
#        #problem_type="single_label_classification"
#    )

#model = XLMRobertaSentimentRegressor("xlm-roberta-base", XLMRobertaConfig.from_pretrained(model_id))
model = XLMRobertaMobius.from_pretrained(model_id)

lora_config = LoraConfig(
    r=128,
    lora_alpha=64,
    target_modules=[
        "query", "key", "value",   # Attention layers
        "intermediate.dense",      # First MLP layer (Backbone only)
        "output.dense"             # Second MLP layer (Backbone only)
    ],
    modules_to_save=["regressor", "intensity_head", "sarcasm_head"], 
    lora_dropout=0.01,
    task_type="SEQ_CLS"
)
#model.roberta = get_peft_model(model.roberta, lora_config)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"trainable params: {trainable_params:,} || all params: {all_params:,} || trainable%: {100 * trainable_params / all_params:.4f}")

# Updated Training Arguments
training_args = TrainingArguments(
    output_dir="./sentiment_results",
    per_device_train_batch_size=64, # Aggressive batch size
    per_device_eval_batch_size=1024,
    learning_rate=1.5e-4,              # Slightly higher for larger batch
    lr_scheduler_type="cosine",
    warmup_steps=100,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=1500,
    save_strategy="steps",
    save_steps=5000,
    fp16=True,
    logging_steps=10,
    remove_unused_columns=False,
)

trainer = GradVacTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
    #data_collator=PassthroughCollator()
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21442.94it/s]
[transformers] XLMRobertaMobius LOAD REPORT from: xlm-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
intensity_head.{0, 3, 5}.weight | MISSING    | 
sarcasm_head.{0, 3, 5}.bias     | MISSING    | 
intensity_head.{0, 3, 5}.bias   | MISSING    | 
sarcasm_head.{0, 3, 5}.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 21,972,482 || all params: 300,754,948 || trainable%: 7.3058


In [ ]:
trainer.train()

# Save locally
#model.save_pretrained("./sentiment_model_lora_final")

Step,Training Loss,Validation Loss,Mae,Raw Mae
1500,0.210469,0.043116,0.422897,0.478640
3000,0.177635,0.042918,0.409365,0.477163
4500,0.162040,0.041783,0.401230,0.464539
6000,0.180790,0.040489,0.393651,0.450585
7500,0.147204,0.040111,0.390119,0.445983


In [ ]:
df_errors = get_error_dataframe(model, trainer.get_eval_dataloader(), tokenizer)

In [ ]:
from util import calculate_pruned_mae, plot_pruning_impact
# 2. Plot the macro view (Are there massive error spikes at 0 and 4?)
plot_error_distribution(df_errors)

# 3. Look at the absolute worst 20 predictions across the board
print("\n=== THE ABSOLUTE WORST PREDICTIONS ===")
worst_overall = get_worst_offenders(df_errors, top_n=20)
display(worst_overall[["true_label", "predicted", "error", "lang", "text"]])
plot_pruning_impact(df_errors, max_exclude_pct=0.2)

In [27]:
check_interference_sampled(trainer, model)

Batch 1/10 Similarity: -0.0987
Batch 2/10 Similarity: -0.0462
Batch 3/10 Similarity: 0.0355
Batch 4/10 Similarity: -0.0218
Batch 5/10 Similarity: 0.2780
Batch 6/10 Similarity: 0.1295
Batch 7/10 Similarity: 0.0927
Batch 8/10 Similarity: 0.2145
Batch 9/10 Similarity: 0.2541
Batch 10/10 Similarity: 0.1245

Average Alignment (EN vs DE): 0.0962


In [28]:
check_interference_balanced(model, dataset, tokenizer, num_batches=10, batch_size=16)

Filter: 100%|██████████| 24450/24450 [00:00<00:00, 50053.14 examples/s]

Reached end of dataset.

Average Alignment (EN vs DE): 0.0000


0

In [40]:
original_df = load_dataset('csv', data_files=file_path, split='train')
prune_and_save_dataset(original_df, df_errors, 7500, output_path="train_lang_pruned.csv")

Pruned 0 samples. New file saved to train_lang_pruned.csv


,id,sentence,label,lang
0,0,Moderner Weihnachtsbaum in weiß\n\nDieses Jahr...,4,deu_Latn
1,1,Passt wie angegossen\n\nTasche kam schnell und...,4,deu_Latn
2,2,Schlechte Qualität\n\nIch habe sehr lange auf ...,1,deu_Latn
3,3,Bestellung nie angekommen\n\n-5 Sterne..... Am...,0,deu_Latn
4,4,Für mich gar nicht gut\n\nMacht das Makeup gar...,1,deu_Latn
...,...,...,...,...
251995,251995,Achtung Finger weg!!!!!\n\nBei uns hat es die ...,0,deu_Latn
251996,251996,Not worth the savings\n\nThird time I've bough...,1,eng_Latn
251997,251997,Nicht zu empfehlen\n\nSchönes Design. Leider d...,0,deu_Latn
251998,251998,deutschlandfahne\n\ndie lieferng war zügig. de...,3,deu_Latn


In [ ]:
import pandas as pd
import torch
import numpy as np
from datasets import Dataset

# 1. Prepare Test Data
df_test = pd.read_csv("test.csv")
test_ds = Dataset.from_pandas(df_test)

def preprocess_test(examples):
    return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)

tokenized_test = test_ds.map(preprocess_test, batched=True)
tokenized_test.set_format("torch")

# 2. Inference
model.eval()
raw_outputs = []
test_loader = torch.utils.data.DataLoader(tokenized_test, batch_size=1024)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        inputs = {k: v.to("cuda:0") for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
        logits = model(**inputs)
        raw_outputs.extend(logits['logits'].squeeze().cpu().numpy())

# 3. Clip and Round
# Clip to ensure results stay within the 0-4 range before rounding
final_preds = np.clip(raw_outputs, 0, 4)
final_preds = np.rint(final_preds).astype(int)

In [ ]:
# 4. Save
df_test["label"] = final_preds
df_test[["id", "label"]].to_csv("submission.csv", index=False)